# EPC Join Investigation Notebook

This notebook walks through every stage of `link_sales_to_epc` step-by-step so you can
inspect the intermediate results at each point and verify correctness.

## Pipeline overview

```
Sales (LR)  ──┐
               ├─ Normalise addresses (_addr_key)
               ├─ EPC semi-join pre-filter  (reduces EPC to sales postcodes)
               ├─ Slim epc_key  (postcode / _addr_key / inspection_date / lmk_key)
               ├─ Phase 1a: join_asof by (postcode, _addr_key)  → addr-matched rows
               ├─ Phase 1b: join_asof by (postcode)             → postcode-only fallback
               ├─ Concat Phase 1a + 1b  →  sales-sized result
               └─ Phase 2: left-join on lmk_key                → attach full EPC features
```

## Configuration

Set `POSTCODE_PREFIX` to limit both sources to a single postcode district (e.g. `"M19"`)
to make all `.collect()` calls fast even on large raw files.  Set `N_SALES` / `N_EPC` to
`None` to use the full filtered dataset.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import polars as pl

# --- Make src importable from the notebook ---
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.config import DATA_DIR
from src.data.pipeline import _addr_key_expr

print(f"Polars version : {pl.__version__}")
print(f"Project root   : {PROJECT_ROOT}")
print(f"Data directory : {DATA_DIR}")

: 

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — edit these to explore different slices of data
# ---------------------------------------------------------------------------

POSTCODE_PREFIX: str = "M19"   # Restrict both datasets to this postcode district
N_SALES: int | None = None      # None = all rows in the postcode slice
N_EPC:   int | None = None      # None = all rows in the postcode slice

RAW_DIR = DATA_DIR / "raw"

SALES_FILE = RAW_DIR / "pp-complete.parquet"
EPC_FILE   = RAW_DIR / "epc_domestic_bulk.parquet"

print(f"Sales file : {SALES_FILE}  (exists={SALES_FILE.exists()})")
print(f"EPC file   : {EPC_FILE}  (exists={EPC_FILE.exists()})")

---
## Stage 0 — Load & filter raw data

We scan the parquet files lazily and apply a postcode prefix filter before collecting,
so only a small district-level slice is materialised in RAM.

In [ ]:
# --- Land Registry (sales) ---
from src.data.sources.land_registry import LandRegistryPricePaid

lr = LandRegistryPricePaid(raw_dir=RAW_DIR)
sales_raw_lf = lr.clean(lr.load())  # LazyFrame: all LR data, cleaned

sales_lf = sales_raw_lf.filter(pl.col("postcode").str.starts_with(POSTCODE_PREFIX))
if N_SALES is not None:
    sales_lf = sales_lf.head(N_SALES)

sales_df = sales_lf.collect()
print(f"Sales rows : {len(sales_df):,}")
print(f"Columns    : {sales_df.columns}")
sales_df.head(5)

In [ ]:
# --- EPC data ---
from src.data.sources.epc import EPCData

epc_source = EPCData(raw_dir=RAW_DIR)
epc_raw_lf = epc_source.clean(epc_source.load())  # LazyFrame: all EPC data, cleaned

epc_lf = epc_raw_lf.filter(pl.col("postcode").str.starts_with(POSTCODE_PREFIX))
if N_EPC is not None:
    epc_lf = epc_lf.head(N_EPC)

epc_df = epc_lf.collect()
print(f"EPC rows   : {len(epc_df):,}")
print(f"Columns    : {epc_df.columns[:15]} ... ({len(epc_df.columns)} total)")
epc_df.select(["postcode", "address1", "inspection_date", "current_energy_rating", "lmk_key"]).head(5)

In [ ]:
# Quick sanity check — unique postcodes in each source
sales_postcodes = set(sales_df["postcode"].unique().to_list())
epc_postcodes   = set(epc_df["postcode"].unique().to_list())
overlap = sales_postcodes & epc_postcodes

print(f"Unique postcodes — Sales: {len(sales_postcodes)}, EPC: {len(epc_postcodes)}, overlap: {len(overlap)}")

# Multiple EPC certs per postcode
epc_per_postcode = epc_df.group_by("postcode").len().sort("len", descending=True)
print("\nEPC certs per postcode (top 5):")
print(epc_per_postcode.head())

---
## Stage 1 — Address normalisation

`_addr_key` is a single uppercase, whitespace-collapsed string built from:
- **Sales side**: `paon` (primary address object name) + `street`  
- **EPC side**: `address1`

A good `_addr_key` match between the two sources identifies the same physical property.

In [ ]:
# --- Sales: add _addr_key ---
sales_with_key = sales_df.with_columns(_addr_key_expr("paon", "street").alias("_addr_key"))

print("Sales _addr_key examples:")
sales_with_key.select(["paon", "street", "_addr_key"]).head(8)

In [ ]:
# --- EPC: add _addr_key from address1 ---
epc_with_key = epc_df.with_columns(
    pl.col("address1")
    .fill_null("")
    .cast(pl.String)
    .str.to_uppercase()
    .str.strip_chars()
    .str.replace_all(r"\s+", " ")
    .alias("_addr_key")
)

print("EPC _addr_key examples:")
epc_with_key.select(["address1", "_addr_key", "postcode"]).head(8)

In [ ]:
# --- Investigate key quality: how many sales have a direct address match in EPC? ---
sales_keys = sales_with_key.select(["postcode", "_addr_key"]).unique()
epc_keys   = epc_with_key.select(["postcode", "_addr_key"]).unique()

addr_matches = sales_keys.join(epc_keys, on=["postcode", "_addr_key"], how="inner")
print(f"Sales (postcode, _addr_key) pairs   : {len(sales_keys):,}")
print(f"EPC   (postcode, _addr_key) pairs   : {len(epc_keys):,}")
print(f"Exact addr+postcode matches         : {len(addr_matches):,}")
print(f"Address match rate (of sales pairs) : {len(addr_matches)/len(sales_keys):.1%}")

---
## Stage 2 — EPC postcode semi-join pre-filter

Before the asof joins, the EPC dataset is semi-joined to only postcodes present in sales.
This eliminates certificates for areas with no matching transactions, shrinking the working
set without changing the schema.

In [ ]:
# Simulate the semi-join
sales_postcodes_df = sales_with_key.select("postcode").unique()

epc_filtered = epc_with_key.join(sales_postcodes_df, on="postcode", how="semi")

print(f"EPC rows before pre-filter : {len(epc_with_key):,}")
print(f"EPC rows after  pre-filter : {len(epc_filtered):,}")
print(f"Reduction                  : {1 - len(epc_filtered)/len(epc_with_key):.1%}")

---
## Stage 3 — Slim EPC key frame

For the Phase-1 asof joins we only need four columns:
`postcode`, `_addr_key`, `inspection_date`, `lmk_key`.

Carrying the full 98-column EPC schema through Phase 1 would waste memory; full features
are attached in Phase 2 (a 1:1 join on `lmk_key`).

The frame must be **sorted on `inspection_date`** for `join_asof` to work correctly.

In [ ]:
epc_id_col = "lmk_key" if "lmk_key" in epc_filtered.columns else None
key_cols = [c for c in ["postcode", "_addr_key", "inspection_date", epc_id_col] if c is not None and c in epc_filtered.columns]

epc_key = (
    epc_filtered
    .select(key_cols)
    .filter(pl.col("inspection_date").is_not_null())
    .sort("inspection_date")
)

print(f"EPC key rows   : {len(epc_key):,}")
print(f"EPC key cols   : {epc_key.columns}")
print(f"Date range     : {epc_key['inspection_date'].min()} → {epc_key['inspection_date'].max()}")
print(f"Null lmk_keys  : {epc_key['lmk_key'].null_count() if 'lmk_key' in epc_key.columns else 'n/a'}")
epc_key.head()

In [ ]:
# --- Prepare sales for asof join ---
# Cast date_of_transfer (Datetime) → _sale_date (Date) to match inspection_date dtype.
# join_asof requires the left frame sorted on the asof key too.
sales_sorted = (
    sales_with_key
    .with_columns(pl.col("date_of_transfer").cast(pl.Date).alias("_sale_date"))
    .sort("_sale_date", nulls_last=True)
)

print(f"Sales schema: {dict(zip(sales_sorted.schema.names(), [str(t) for t in sales_sorted.schema.dtypes()]))}")
print(f"\n_sale_date range: {sales_sorted['_sale_date'].min()} → {sales_sorted['_sale_date'].max()}")

sales_sorted.select(["transaction_id", "postcode", "_addr_key", "_sale_date", "price"]).head(5)

---
## Stage 4a — Phase 1a: join_asof by (postcode, _addr_key)

For each sale, find the EPC certificate at the **same postcode AND normalised address**
whose `inspection_date` is closest (nearest) to the sale's `_sale_date`.

This is O(n log m) — no fan-out, always sales-sized output.

Rows where `lmk_key` is null after the join had no matching `(postcode, _addr_key)` pair
and will fall through to Phase 1b.

In [ ]:
import warnings

# suppress the benign Polars sortedness warning for by-grouped asof joins
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning, message="Sortedness")
    phase1a = sales_sorted.join_asof(
        epc_key.lazy().collect(),  # materialised for step-by-step inspection
        left_on="_sale_date",
        right_on="inspection_date",
        by=["postcode", "_addr_key"],
        strategy="nearest",
        suffix="_epc",
    )

null_col = epc_id_col or "inspection_date"
n_matched   = phase1a.filter(pl.col(null_col).is_not_null()).shape[0]
n_unmatched = phase1a.filter(pl.col(null_col).is_null()).shape[0]

print(f"Phase 1a output rows : {len(phase1a):,}  (should equal sales rows: {len(sales_sorted):,})")
print(f"  Addr+postcode match : {n_matched:,}  ({n_matched/len(phase1a):.1%})")
print(f"  Unmatched (null)    : {n_unmatched:,}  ({n_unmatched/len(phase1a):.1%})")

phase1a.select(["transaction_id", "postcode", "_addr_key", "_sale_date", null_col, "inspection_date"]).head(8)

In [ ]:
# Inspect some address-matched rows
addr_matched = phase1a.filter(pl.col(null_col).is_not_null())

print("Sample address-matched sales:")
addr_matched.select([
    "transaction_id", "postcode", "_addr_key", "_sale_date", "inspection_date", null_col
]).head(8)

In [ ]:
# Check the temporal gap between sale and the chosen EPC certificate
date_gap = addr_matched.with_columns(
    (pl.col("_sale_date") - pl.col("inspection_date")).dt.total_days().abs().alias("days_gap")
)

print("Temporal gap (sale_date - inspection_date) for addr-matched rows:")
print(date_gap["days_gap"].describe())

In [ ]:
# --- Split matched vs unmatched; strip EPC columns from unmatched before Phase 1b ---
sales_schema_cols = set(sales_sorted.columns)
epc_added_by_1a   = [c for c in phase1a.columns if c not in sales_schema_cols]

addr_unmatched = phase1a.filter(pl.col(null_col).is_null()).drop(epc_added_by_1a)

print(f"Matched rows   : {len(addr_matched):,}")
print(f"Unmatched rows : {len(addr_unmatched):,}")
print(f"EPC cols added by Phase 1a (dropped from unmatched): {epc_added_by_1a}")

---
## Stage 4b — Phase 1b: join_asof by (postcode) — postcode-only fallback

For the sales that Phase 1a could not match (no `(postcode, _addr_key)` pair in EPC),
we fall back to a looser join: find the EPC certificate in the **same postcode** with the
nearest `inspection_date`.

This is lower confidence than the address match but avoids leaving sales with no EPC data
at all when the postcode has EPC coverage.

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning, message="Sortedness")
    phase1b = addr_unmatched.join_asof(
        epc_key.lazy().collect(),
        left_on="_sale_date",
        right_on="inspection_date",
        by=["postcode"],
        strategy="nearest",
        suffix="_epc",
    )

n_1b_matched   = phase1b.filter(pl.col(null_col).is_not_null()).shape[0]
n_1b_unmatched = phase1b.filter(pl.col(null_col).is_null()).shape[0]

print(f"Phase 1b output rows  : {len(phase1b):,}")
print(f"  Postcode match      : {n_1b_matched:,}  ({n_1b_matched/max(len(phase1b),1):.1%})")
print(f"  Still unmatched     : {n_1b_unmatched:,}  ({n_1b_unmatched/max(len(phase1b),1):.1%})")

phase1b.select(["transaction_id", "postcode", "_addr_key", "_sale_date", null_col, "inspection_date"]).head(5)

---
## Stage 5 — Concat Phase 1a + 1b and drop working columns

`diagonal_relaxed` concat handles the schema difference between the two frames:
Phase 1b gains an `_addr_key_epc` column (the EPC address of the postcode-matched record)
that Phase 1a does not have.  Missing columns are filled with `null`.

In [ ]:
combined = pl.concat([addr_matched, phase1b], how="diagonal_relaxed")

print(f"Combined rows : {len(combined):,}  (should equal sales rows: {len(sales_sorted):,})")
print(f"Columns       : {combined.columns}")

# Sanity: no duplicate transaction_ids introduced?
n_unique_txn = combined["transaction_id"].n_unique()
print(f"\nUnique transaction_ids : {n_unique_txn:,} / {len(combined):,}  ",
      "✅ no duplicates" if n_unique_txn == len(combined) else "❌ DUPLICATES DETECTED")

In [ ]:
# Drop working columns (_addr_key, _addr_key_epc, _sale_date)
drop_cols = [c for c in ["_addr_key", "_addr_key_epc", "_sale_date"] if c in combined.columns]
phase1_result = combined.drop(drop_cols)

print(f"Columns after drop : {phase1_result.columns}")
phase1_result.select(["transaction_id", "postcode", "price", "inspection_date", null_col]).head(5)

In [ ]:
# Overall Phase-1 match rate summary
total = len(phase1_result)
addr_hits    = n_matched
postcode_hits = n_1b_matched
no_hits      = n_1b_unmatched

print("=" * 50)
print(f"Phase-1 EPC match summary ({total:,} sales)")
print("=" * 50)
print(f"  Address + postcode match : {addr_hits:,} ({addr_hits/total:.1%})")
print(f"  Postcode-only match      : {postcode_hits:,} ({postcode_hits/total:.1%})")
print(f"  No EPC match             : {no_hits:,} ({no_hits/total:.1%})")

---
## Stage 6 — Phase 2: 1:1 feature join on lmk_key

Phase 1 produced a sales-sized frame with only `lmk_key` as the EPC identifier.
Phase 2 is a plain left-join that attaches the full 90+ EPC feature columns on that key.

Because the result entering Phase 2 is already one-row-per-sale, this join cannot fan out —
it is a simple 1:1 lookup.

In [ ]:
if epc_id_col is not None:
    # Drop columns that are already in phase1_result to avoid spurious _epc suffixes
    epc_feature_drop = [c for c in ["postcode", "_addr_key", "inspection_date"] if c in epc_filtered.columns]
    epc_features = epc_filtered.drop(epc_feature_drop)

    print(f"EPC feature cols for Phase 2: {len(epc_features.columns)} columns")
    print(f"Joining on: {epc_id_col}")

    final = phase1_result.join(epc_features, on=epc_id_col, how="left", suffix="_epc")
    print(f"\nFinal output rows    : {len(final):,}")
    print(f"Final output columns : {len(final.columns)}")
else:
    print("No lmk_key — Phase 2 skipped; Phase-1 result IS the final result.")
    final = phase1_result

In [ ]:
# Verify no fan-out occurred in Phase 2
n_final_unique = final["transaction_id"].n_unique()
print(f"Final rows             : {len(final):,}")
print(f"Unique transaction_ids : {n_final_unique:,}")
print("✅ no duplicates" if n_final_unique == len(final) else "❌ DUPLICATES DETECTED")

# Show selected EPC columns in the final result
epc_preview_cols = [
    c for c in [
        "transaction_id", "postcode", "price", "inspection_date",
        "current_energy_rating", "current_energy_efficiency",
        "total_floor_area", "property_type"
    ] if c in final.columns
]
final.select(epc_preview_cols).head(8)

---
## Stage 7 — Summary statistics & quality checks

In [ ]:
# EPC rating distribution in matched sales
if "current_energy_rating" in final.columns:
    rating_dist = (
        final
        .group_by("current_energy_rating")
        .len()
        .sort("current_energy_rating")
    )
    print("EPC rating distribution (null = no match):")
    print(rating_dist)

In [ ]:
# Price vs energy efficiency (for matched rows)
if "current_energy_efficiency" in final.columns:
    matched_only = final.filter(pl.col("current_energy_efficiency").is_not_null())
    corr = matched_only.select(
        pl.corr("price", "current_energy_efficiency").alias("price_vs_efficiency_corr")
    )
    print(f"Pearson correlation (price vs current_energy_efficiency): {corr.item():.4f}")
    print(f"Based on {len(matched_only):,} matched rows")

In [ ]:
# Null check on key columns
null_summary = {
    col: final[col].null_count()
    for col in ["transaction_id", "postcode", "price", "current_energy_rating", "inspection_date"]
    if col in final.columns
}
print("Null counts in final output:")
for col, n in null_summary.items():
    pct = n / len(final)
    print(f"  {col:<30} {n:>6,} ({pct:.1%})")

---
## Stage 8 — Deep-dive: inspect a specific property

Use the cell below to look at a particular transaction and trace which EPC certificate
was selected and why.

In [ ]:
# Pick a postcode to investigate in detail
INVESTIGATE_POSTCODE = final["postcode"][0]  # Change to any postcode of interest

print(f"=== Investigating postcode: {INVESTIGATE_POSTCODE} ===")

# All sales in this postcode
sales_in_pc = sales_df.filter(pl.col("postcode") == INVESTIGATE_POSTCODE)
print(f"\nSales in {INVESTIGATE_POSTCODE} ({len(sales_in_pc)} rows):")
print(sales_in_pc.select(["transaction_id", "paon", "street", "date_of_transfer", "price"]))

# All EPC certs in this postcode
epc_in_pc = epc_df.filter(pl.col("postcode") == INVESTIGATE_POSTCODE)
print(f"\nEPC certs in {INVESTIGATE_POSTCODE} ({len(epc_in_pc)} rows):")
print(epc_in_pc.select(["address1", "inspection_date", "current_energy_rating", "lmk_key"]))

# Final matched result for this postcode
final_in_pc = final.filter(pl.col("postcode") == INVESTIGATE_POSTCODE)
print(f"\nFinal matched rows in {INVESTIGATE_POSTCODE} ({len(final_in_pc)} rows):")
print(final_in_pc.select(epc_preview_cols))

---
## Stage 9 — (Optional) Run via `run_pipeline` and compare

The cell below runs the full `run_pipeline` function (which executes via streaming sinks)
on the same filtered input data and compares the result to the step-by-step output above.

In [ ]:
from src.data.pipeline import run_pipeline
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    pipeline_result = run_pipeline(
        sales=sales_df.lazy(),
        epc=epc_df.lazy(),
        output_name="investigate_epc_join",
        output_dir=Path(tmp),
        fmt="parquet",
    )

print(f"run_pipeline rows    : {len(pipeline_result):,}")
print(f"Step-by-step rows    : {len(final):,}")
print(f"Row counts match     : {len(pipeline_result) == len(final)}")

# Compare transaction_id sets
pipeline_txns = set(pipeline_result["transaction_id"].to_list())
manual_txns   = set(final["transaction_id"].to_list())
print(f"Transaction ID sets identical: {pipeline_txns == manual_txns}")